# 05 - Performance Tuning & Operations

Single dataset: NYC Taxi.

Coverage:
- AQE: coalescing, runtime join switching, skew mitigation, and limits
- Broadcast variables and accumulators (including retry semantics)
- Dynamic resource allocation requirements and trade-offs
- Speculative execution for stragglers
- Fault tolerance: retries, lineage recomputation, DAG recovery, checkpointing


## Setup
All experiments are configuration-driven and timing-measured.


In [ ]:
from pathlib import Path
import os, time
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark import TaskContext


def _root():
    env = os.getenv("SPARK_TUNING_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Set SPARK_TUNING_PROJECT_ROOT")

ROOT = _root()
DATA = Path(os.getenv("SPARK_TUNING_DATA_DIR", str(ROOT / "data"))).resolve()
TAXI = DATA / "taxi"
TMP = DATA / "tmp"
CP = TMP / "checkpoints_05"
os.makedirs(TMP, exist_ok=True)
os.makedirs(CP, exist_ok=True)

print("ROOT", ROOT)
print("DATA", DATA)
print("TAXI", TAXI)


In [ ]:
taxi = spark.read.parquet(str(TAXI))
trips = taxi.select("VendorID","PULocationID","DOLocationID","payment_type","fare_amount","tip_amount","total_amount")
zone = taxi.groupBy("PULocationID","DOLocationID").agg(F.count("*").alias("trip_count"), F.avg("total_amount").alias("avg_total"))
loc = taxi.select("PULocationID","RatecodeID","Airport_fee").dropDuplicates(["PULocationID"])

print("app", spark.sparkContext.applicationId)
print("master", spark.sparkContext.master)
print("rows", taxi.count())
print("zone rows", zone.count())
print("loc rows", loc.count())


## Helpers


In [ ]:
def run_and_report(label, fn):
    sc = spark.sparkContext
    tr = sc.statusTracker()
    gid = f"nb05_{int(time.time()*1000)}"
    sc.setJobGroup(gid, label)
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    jobs = list(tr.getJobIdsForGroup(gid))
    stages = set()
    for j in jobs:
        ji = tr.getJobInfo(j)
        if ji is not None:
            stages.update(list(ji.stageIds))
    print(f"{label} -> {dt:.2f}s jobs={jobs} stages={sorted(stages)}")
    for s in sorted(stages):
        si = tr.getStageInfo(s)
        if si is not None:
            print(f"  stage {s}: tasks total={si.numTasks} done={si.numCompletedTasks} failed={si.numFailedTasks}")
    sc.setJobGroup("", "")
    return out, dt


def show_nodes(df, keys=("AdaptiveSparkPlan","CustomShuffleReader","SortMergeJoin","BroadcastHashJoin","Exchange","BatchEvalPython")):
    p = df._jdf.queryExecution().executedPlan().toString()
    for ln in p.splitlines():
        if any(k in ln for k in keys):
            print(ln.strip())


def set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20):
    spark.conf.set("spark.sql.adaptive.enabled", str(enabled).lower())
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", str(coalesce).lower())
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", str(skew).lower())
    spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", str(adaptive_broadcast_mb*1024*1024))


def exec_count():
    return len(spark.sparkContext.statusTracker().getExecutorInfos())


## 1) AQE deep dive
AQE can coalesce partitions, switch join strategy at runtime, and mitigate skew. It is not a fix for Python UDF overhead or incorrect query semantics.


In [ ]:
def q_join():
    return trips.join(zone, on=["PULocationID","DOLocationID"], how="inner").groupBy("PULocationID").agg(F.count("*").alias("rows"), F.avg("total_amount").alias("avg_total"))

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.shuffle.partitions", "200")
set_aqe(enabled=False)
q_off = q_join()
show_nodes(q_off)
_, t_off = run_and_report("AQE OFF", lambda: q_off.count())

set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20)
q_on = q_join()
show_nodes(q_on)
_, t_on = run_and_report("AQE ON", lambda: q_on.count())
show_nodes(q_on)
print("AQE off", round(t_off,2), "AQE on", round(t_on,2), "factor", round(t_off/max(t_on,1e-9),2))


In [ ]:
spark.conf.set("spark.sql.shuffle.partitions", "400")

set_aqe(enabled=False)
co_off = trips.groupBy("PULocationID","DOLocationID").agg(F.count("*").alias("c"), F.avg("total_amount").alias("a"))
_, t_co_off = run_and_report("coalescing exp AQE OFF", lambda: co_off.count())

set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20)
co_on = trips.groupBy("PULocationID","DOLocationID").agg(F.count("*").alias("c"), F.avg("total_amount").alias("a"))
_, t_co_on = run_and_report("coalescing exp AQE ON", lambda: co_on.count())
show_nodes(co_on)
print("coalesce off", round(t_co_off,2), "on", round(t_co_on,2))


In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=32)
runtime_small = zone.filter(F.col("trip_count") > 50000)
switch_q = trips.join(runtime_small, on=["PULocationID","DOLocationID"], how="inner").groupBy("PULocationID").agg(F.sum("trip_count").alias("s"))
print("join switching candidate pre-action")
show_nodes(switch_q)
_, t_switch = run_and_report("runtime join switching", lambda: switch_q.count())
print("post-action nodes")
show_nodes(switch_q)
print("switch seconds", round(t_switch,2))


In [ ]:
sk_left = trips.withColumn("k", F.when((F.col("PULocationID")%20)==0, F.col("PULocationID")).otherwise(F.lit(0))).select("k","total_amount")
sk_right = trips.withColumn("k", F.when((F.col("PULocationID")%20)==0, F.col("PULocationID")).otherwise(F.lit(0))).select("k").dropDuplicates(["k"])

set_aqe(enabled=True, coalesce=True, skew=False, adaptive_broadcast_mb=20)
no_sk = sk_left.join(sk_right, on="k", how="inner")
_, t_no_sk = run_and_report("AQE skew OFF", lambda: no_sk.count())

set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20)
yes_sk = sk_left.join(sk_right, on="k", how="inner")
_, t_yes_sk = run_and_report("AQE skew ON", lambda: yes_sk.count())
print("skew off", round(t_no_sk,2), "skew on", round(t_yes_sk,2))


In [ ]:
@udf("double")
def tip_ratio_py(t, f):
    if f is None or f == 0 or t is None:
        return 0.0
    return float(t)/float(f)

set_aqe(enabled=True, coalesce=True, skew=True, adaptive_broadcast_mb=20)
b = trips.filter(F.col("fare_amount")>0).withColumn("r", F.col("tip_amount")/F.col("fare_amount"))
u = trips.filter(F.col("fare_amount")>0).withColumn("r", tip_ratio_py(F.col("tip_amount"), F.col("fare_amount")))
_, t_b = run_and_report("AQE builtin", lambda: b.groupBy("PULocationID").agg(F.avg("r")).count())
_, t_u = run_and_report("AQE python udf", lambda: u.groupBy("PULocationID").agg(F.avg("r")).count())
show_nodes(u)
print("builtin", round(t_b,2), "python_udf", round(t_u,2), "factor", round(t_u/max(t_b,1e-9),2))


## 2) Broadcast variables and accumulators
Broadcast semantics: read-only side data per executor with explicit lifecycle.
Accumulator semantics: executor-side adds, driver-side read, retry can double-count.


In [ ]:
sc = spark.sparkContext
hot = sc.broadcast(set([132,138,170,237,236,161]))

def mark_hot(rows):
    h = hot.value
    for r in rows:
        yield (r.PULocationID in h, 1)

flags = trips.select("PULocationID").rdd.mapPartitions(mark_hot).toDF(["is_hot","one"])
_, t_hot = run_and_report("broadcast variable usage", lambda: flags.groupBy("is_hot").agg(F.sum("one")).collect())
print("seconds", round(t_hot,2))
hot.unpersist(); hot.destroy()


In [ ]:
acc = sc.longAccumulator("high_fare")

def f_acc(row):
    if row.total_amount is not None and row.total_amount > 100:
        acc.add(1)
    return row

_, t_acc = run_and_report("acc basic", lambda: trips.select("total_amount").rdd.map(f_acc).count())
print("seconds", round(t_acc,2), "value", acc.value)

acc_retry = sc.longAccumulator("retry_sensitive")
def flaky(idx, it):
    tc = TaskContext.get()
    for x in it:
        acc_retry.add(1)
        if idx == 0 and tc.attemptNumber() == 0 and x < 10:
            raise RuntimeError("inject retry")
        yield x

try:
    _, t_retry = run_and_report("acc retry demo", lambda: sc.parallelize(range(2000),8).mapPartitionsWithIndex(flaky).count())
    print("retry seconds", round(t_retry,2))
except Exception as e:
    print("retry path raised", type(e).__name__)
print("retry-sensitive value", acc_retry.value, "(may exceed logical count)")


## 3) Dynamic resource allocation
Required controls (startup scope in most deployments):
- `spark.dynamicAllocation.enabled`
- `spark.shuffle.service.enabled` (or equivalent tracking mode)
- min/initial/max executors

Trade-off:
- better cluster utilization and elasticity
- possible latency variance and warm-up overhead


In [ ]:
for k in [
    "spark.dynamicAllocation.enabled",
    "spark.dynamicAllocation.minExecutors",
    "spark.dynamicAllocation.initialExecutors",
    "spark.dynamicAllocation.maxExecutors",
    "spark.dynamicAllocation.executorIdleTimeout",
    "spark.shuffle.service.enabled",
]:
    print(k, "=", spark.conf.get(k, "<unset>"))

before = exec_count()
spark.conf.set("spark.dynamicAllocation.enabled", "true")
work = trips.repartition(200,"PULocationID").groupBy("PULocationID").agg(F.avg("total_amount").alias("a"))
_, t_dyn = run_and_report("dynamic allocation runtime toggle", lambda: work.count())
after = exec_count()
print("seconds", round(t_dyn,2), "executors before", before, "after", after)
print("If unchanged: current mode requires startup-time enablement")


## 4) Speculative execution
Goal: mitigate stragglers by launching duplicate attempts.
Cost: duplicated work. Benefit depends on spare capacity.


In [ ]:
def straggler(spec_on):
    spark.conf.set("spark.speculation", str(spec_on).lower())
    def slow(i, it):
        if i == 0:
            time.sleep(6)
        for x in it:
            yield x
    r = spark.sparkContext.parallelize(range(3_000_000),24).mapPartitionsWithIndex(slow)
    return run_and_report(f"speculation={spec_on}", lambda: r.map(lambda x: x*2).count())[1]

try:
    t0 = straggler(False)
    t1 = straggler(True)
    print("spec off", round(t0,2), "spec on", round(t1,2), "factor", round(t0/max(t1,1e-9),2))
except Exception as e:
    print("spec experiment issue", e)


## 5) Fault tolerance: retries, lineage, DAG recovery, checkpointing


In [ ]:
def flaky_once(idx, it):
    tc = TaskContext.get()
    if idx == 0 and tc.attemptNumber() == 0:
        raise RuntimeError("transient partition failure")
    for x in it:
        yield x

r = spark.sparkContext.parallelize(range(1_000_000),12).mapPartitionsWithIndex(flaky_once)
_, t_ft = run_and_report("stage retry recovery", lambda: r.count())
print("retry recovered seconds", round(t_ft,2))


In [ ]:
spark.sparkContext.setCheckpointDir(str(CP))
base = spark.sparkContext.parallelize(range(5_000_000),24)
lin = base.map(lambda x: x+1).map(lambda x: x*2).filter(lambda x: x%3!=0).map(lambda x: x-7)

def dbg(rdd):
    d = rdd.toDebugString()
    return d.decode("utf-8","ignore") if isinstance(d, bytes) else d

print("lineage before checkpoint")
print(dbg(lin)[:500])
_, t_before = run_and_report("lineage action before checkpoint", lambda: lin.count())
lin.checkpoint()
_, t_cp = run_and_report("checkpoint materialization", lambda: lin.count())
print("lineage after checkpoint")
print(dbg(lin)[:500])
print("isCheckpointed", lin.isCheckpointed())
print("checkpointFile", lin.getCheckpointFile())
print("before", round(t_before,2), "after checkpoint", round(t_cp,2))


## Operations runbook
- Tail latency spikes: inspect stragglers, speculation, skew
- Throughput collapse: inspect exchange volume, partition sizing, join strategy
- Repeated failures: isolate transient vs deterministic errors with attempt-aware logs
- Excessive recomputation: introduce checkpoint boundaries

## Key takeaways
- AQE is high-leverage but bounded by query semantics and Python boundaries.
- Broadcast and accumulators need lifecycle and retry-aware handling.
- Dynamic allocation is powerful but config-heavy and often startup-scoped.
- Fault tolerance in Spark is lineage-first; checkpointing is explicit lineage truncation.
